# Phase 2 — Feature Engineering & Preprocessing Pipeline

**Status:** Complete · **Deliverable:** DVC-versioned, tested feature dataset (t+1 / t+6 horizons)

**Rule:** a feature must be available at or before forecast origin `t`. Raw sensors are usable only lagged; calendar features are always usable.

Canonical implementation: `src/` + `config/`, 25 passing tests. This notebook keeps only the cells that produced a design decision or a finding.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from config.paths import RAW_DATA_PATH

df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
print(df_raw.shape, "| monotonic:", df_raw["date"].is_monotonic_increasing)
df_raw.head(2)

(19735, 29) | monotonic: True


,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.7900,19.79,44.73,19.0,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.7225,19.79,44.79,19.0,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195


## 1. Lag Features
`Appliances_lag_{1..6, 144}` — short lags (10–60 min) + daily echo (24h), from Phase 1 ACF/PACF. `.shift(N)`, N>0 only → strictly past.

In [2]:
from config.features import LAG_STEPS

df = df_raw.copy()
for lag in LAG_STEPS:
    df[f"Appliances_lag_{lag}"] = df["Appliances"].shift(lag)

df[["date", "Appliances", "Appliances_lag_1", "Appliances_lag_6", "Appliances_lag_144"]].head(3)

,date,Appliances,Appliances_lag_1,Appliances_lag_6,Appliances_lag_144
0,2016-01-11 17:00:00,60,NaN,NaN,NaN
1,2016-01-11 17:10:00,60,60.0,NaN,NaN
2,2016-01-11 17:20:00,50,60.0,NaN,NaN


## 2. Rolling Stats — Deliberate Leak Check
`Appliances_roll{6,18}_{mean,std}` (~1hr / ~3hr), includes current row `Appliances[t]` (not a leak — it's observed at origin).

Risk: `center=True`. Triggered below, then discarded.

In [3]:
row = 100
roll_leaky = df["Appliances"].rolling(window=6, center=True).mean()
roll_correct = df["Appliances"].rolling(window=6).mean()

print(df.loc[row - 2:row + 3, ["date", "Appliances"]])
print(f"\nLeaky mean (uses 3 future rows): {roll_leaky.loc[row]:.2f}")
print(f"Correct mean (backward-only):     {roll_correct.loc[row]:.2f}")

                   date  Appliances
98  2016-01-12 09:20:00          50
99  2016-01-12 09:30:00          30
100 2016-01-12 09:40:00          40
101 2016-01-12 09:50:00          30
102 2016-01-12 10:00:00         260
103 2016-01-12 10:10:00         500

Leaky mean (uses 3 future rows): 78.33
Correct mean (backward-only):     48.33


**Finding:** leaky mean 78.33 vs. correct 48.33 — three future rows pulled in.

**Decision:** `add_rolling_features()` exposes no `center` parameter at all (not `=False` — omitted entirely: both values aren't valid, so don't imply they are). Also undefined at serving time — no future row exists in a live buffer. Proven in code by `test_rolling.py` (mutate future row → past feature unchanged).

In [4]:
from src.features.rolling import add_rolling_features

df = add_rolling_features(df, column="Appliances", windows=[6, 18])
df[["date", "Appliances", "Appliances_roll6_mean", "Appliances_roll6_std",
    "Appliances_roll18_mean", "Appliances_roll18_std"]].head(5)

,date,Appliances,Appliances_roll6_mean,Appliances_roll6_std,Appliances_roll18_mean,Appliances_roll18_std
0,2016-01-11 17:00:00,60,NaN,NaN,NaN,NaN
1,2016-01-11 17:10:00,60,NaN,NaN,NaN,NaN
2,2016-01-11 17:20:00,50,NaN,NaN,NaN,NaN
3,2016-01-11 17:30:00,50,NaN,NaN,NaN,NaN
4,2016-01-11 17:40:00,60,NaN,NaN,NaN,NaN


## 3. Time / Cyclical Features
Raw `hour_of_day`, `day_of_week`, `is_weekend` for tree models. Cyclical `minute_of_day_sin/cos` uses **period 1440** (not hour-of-day/24 — data resolution is 10 min). `day_of_week_sin/cos` period 7. No leakage risk — calendar position is always known ahead.

In [5]:
from src.features.temporal import add_time_features

df = add_time_features(df, date_column="date")
df[["date", "hour_of_day", "day_of_week", "is_weekend",
    "minute_of_day_sin", "minute_of_day_cos"]].head(3)

,date,hour_of_day,day_of_week,is_weekend,minute_of_day_sin,minute_of_day_cos
0,2016-01-11 17:00:00,17,0,0,-0.965926,-0.258819
1,2016-01-11 17:10:00,17,0,0,-0.976296,-0.216440
2,2016-01-11 17:20:00,17,0,0,-0.984808,-0.173648


## 4. Targets
`target_t1 = Appliances.shift(-1)`, `target_t6 = Appliances.shift(-6)`. Negative shift here is correct, not leakage — leakage is features-vs-target, not past-vs-future.

Not merged with `add_lag_features` despite similar code — different semantics (feature vs. label) shouldn't hide behind one generic helper. Positive-only guards enforce the shift-sign convention.

In [6]:
from src.features.targets import add_targets

df = add_targets(df, column="Appliances", horizons=[1, 6])
df[["date", "Appliances", "target_t1", "target_t6"]].tail(5)

,date,Appliances,target_t1,target_t6
19730,2016-05-27 17:20:00,100,90.0,NaN
19731,2016-05-27 17:30:00,90,270.0,NaN
19732,2016-05-27 17:40:00,270,420.0,NaN
19733,2016-05-27 17:50:00,420,430.0,NaN
19734,2016-05-27 18:00:00,430,NaN,NaN


## 5. Time-Aware Train / Val / Test Split
Phase 1 STL: daily-cycle amplitude declines monotonically (Early ≈94 → Middle ≈74 → Late ≈60) — a regime shift, not noise. Drove:
- Test ≥ 4 full weeks (enough weekly cycles to trust a metric, not luck on the low-amplitude "Late" regime).
- Val sits in the *middle* regime on purpose, so Phase 3 tuning can't quietly overfit to a regime test won't match.
- Boundaries are calendar-day, not row-percentage (keeps daily/weekly cycle features clean at the edges).

| Partition | Range | Days |
|---|---|---|
| Train | 2016-01-11 → 2016-04-17 | 98 |
| Val | 2016-04-18 → 2016-04-29 | 12 |
| Test | 2016-04-30 → 2016-05-27 | 28 |

Half-open: `train: date<train_end`, `val: train_end<=date<val_end`, `test: date>=val_end`.

In [7]:
from config.features import SPLIT_TRAIN_END, SPLIT_VAL_END
from src.data.split import create_time_masks

train_end, val_end = pd.Timestamp(SPLIT_TRAIN_END), pd.Timestamp(SPLIT_VAL_END)
masks = create_time_masks(df, train_end=train_end, val_end=val_end)

for name, mask in masks.items():
    subset = df.loc[mask]
    n_days = (subset["date"].max() - subset["date"].min()).days + 1
    print(f"{name}: {len(subset)} rows, {n_days} days, {subset['date'].min()} -> {subset['date'].max()}")

train: 14010 rows, 98 days, 2016-01-11 17:00:00 -> 2016-04-17 23:50:00
val: 1728 rows, 12 days, 2016-04-18 00:00:00 -> 2016-04-29 23:50:00
test: 3997 rows, 28 days, 2016-04-30 00:00:00 -> 2016-05-27 18:00:00


## 6. Scaler Leakage — Demonstrated on `T_out`
Full-dataset fit is a **distributional** leak (no row copied, but train stats "peek" at future mean/variance) — the kind that only surfaces once the real distribution shifts.

In [8]:
from sklearn.preprocessing import StandardScaler

feature = "T_out"
scaler_leaky = StandardScaler().fit(df[[feature]])
scaler_correct = StandardScaler().fit(df.loc[masks["train"], [feature]])

sample = df.loc[masks["train"], feature].iloc[0]
print(f"Full-data mean={scaler_leaky.mean_[0]:.2f} | train-only mean={scaler_correct.mean_[0]:.2f}")
print(f"Same raw value {sample}: leaky-scaled={(sample-scaler_leaky.mean_[0])/scaler_leaky.scale_[0]:.2f}, "
      f"correct-scaled={(sample-scaler_correct.mean_[0])/scaler_correct.scale_[0]:.2f}")

Full-data mean=7.41 | train-only mean=5.72
Same raw value 6.6: leaky-scaled=-0.15, correct-scaled=0.21


**Finding:** 7.41 (full) vs 5.72 (train-only) mean — real winter→spring shift. Same raw value (6.6°C) flips sign: below-average under the leaky scaler, above-average under the correct one.

**Decision:** `fit_scaler(df, columns, train_mask)` requires a mask (can't be pointed at anything but a subset). `transform_with_scaler(df, scaler, columns)` takes no mask — reusable unmodified at serving time. No single "auto-decide fit vs transform" function — too easy to accidentally refit and reintroduce skew. `SCALED_COLUMNS` is an explicit whitelist, excluding binary/cyclical/ordinal columns.

In [9]:
from config.features import SCALED_COLUMNS
from src.preprocessing.scaling import fit_scaler, transform_with_scaler

scaler = fit_scaler(df, SCALED_COLUMNS, masks["train"])
df_train_scaled = transform_with_scaler(df.loc[masks["train"]], scaler, SCALED_COLUMNS)
df_train_scaled[SCALED_COLUMNS].describe().loc[["mean", "std"]]

,Appliances_lag_1,Appliances_lag_2,Appliances_lag_3,Appliances_lag_4,Appliances_lag_5,Appliances_lag_6,Appliances_lag_144,Appliances_roll6_mean,Appliances_roll6_std,Appliances_roll18_mean,Appliances_roll18_std
mean,3.144668e-17,1.014481e-17,-1.724741e-17,4.870206e-17,-6.189662e-17,5.987150e-17,-1.024871e-18,4.058795e-18,8.929348e-17,-1.289772e-16,-7.312096e-17
std,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00,1.000036e+00


## 7. Lesson: Does `StandardScaler.fit()` Raise on NaN?
Assumed yes (would need a NaN mask before fitting) — **verified false**: it silently skips NaNs per-column and leaves NaN on `.transform()`. Original design (fit directly on the raw train slice) was already correct; no extra step added. Lesson: verify library behavior before designing around an assumption.

In [10]:
test_scaler = StandardScaler()
try:
    test_scaler.fit(df.loc[masks["train"], SCALED_COLUMNS])
    print("fit() succeeded despite NaNs — no error raised")
except ValueError as e:
    print("fit() raised:", e)

fit() succeeded despite NaNs — no error raised


## 8. End-to-End Pipeline
`run_pipeline()` (`src/pipeline.py`) composes all steps above and returns a `PipelineResult` dataclass (named fields, not a dict — catches typos at dev time).

In [11]:
from src.pipeline import run_pipeline

df_raw_fresh = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
result = run_pipeline(df_raw_fresh)

for name in ["train_t1", "val_t1", "test_t1", "train_t6", "val_t6", "test_t6"]:
    data = getattr(result, name)
    print(f"{name}: {data.shape}, NaNs={data.isna().sum().sum()}")

train_t1: (13866, 20), NaNs=0
val_t1: (1728, 20), NaNs=0
test_t1: (3996, 20), NaNs=0
train_t6: (13866, 20), NaNs=0
val_t6: (1728, 20), NaNs=0
test_t6: (3991, 20), NaNs=0


**Scoped `dropna`:** `build_horizon_dataset()` drops NaN only in that horizon's own feature+target columns — never a blanket `dropna()`. Below: train/val counts match across horizons (shared head burn-in only); test differs by exactly 5 rows (t+6's extra tail loss).

In [12]:
for partition in ["train", "val", "test"]:
    t1, t6 = len(getattr(result, f"{partition}_t1")), len(getattr(result, f"{partition}_t6"))
    print(f"{partition}: t1={t1}, t6={t6}, diff={t1 - t6}")

train: t1=13866, t6=13866, diff=0
val: t1=1728, t6=1728, diff=0
test: t1=3996, t6=3991, diff=5


## 9. Persist DVC-Versioned Artifacts
Six per-partition files (not combined CSVs) — makes the split boundary structural, so Phase 3 can't accidentally train on val/test rows. Regenerated from `src/pipeline.py`, not notebook cells, so the artifact matches the tested code.

In [13]:
import joblib, json

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for name in ["train_t1", "val_t1", "test_t1", "train_t6", "val_t6", "test_t6"]:
    getattr(result, name).to_csv(PROCESSED_DIR / f"{name}.csv", index=False)

joblib.dump(result.scaler, PROCESSED_DIR / "scaler_train_fit.joblib")
with open(PROCESSED_DIR / "split_boundaries.json", "w") as f:
    json.dump({"train_end": SPLIT_TRAIN_END, "val_end": SPLIT_VAL_END}, f, indent=2)

print("Saved:", [p.name for p in sorted(PROCESSED_DIR.iterdir())])

Saved: ['scaler_train_fit.joblib', 'split_boundaries.json', 'test_t1.csv', 'test_t6.csv', 'train_t1.csv', 'train_t6.csv', 'val_t1.csv', 'val_t6.csv']


In [11]:
import pandas as pd
import joblib
import json
from pathlib import Path

from config.paths import RAW_DATA_PATH
from config.features import SPLIT_TRAIN_END, SPLIT_VAL_END
from src.pipeline import run_pipeline

df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
result = run_pipeline(df_raw)

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"

datasets = {
    "train_t1": result.train_t1, "val_t1": result.val_t1, "test_t1": result.test_t1,
    "train_t6": result.train_t6, "val_t6": result.val_t6, "test_t6": result.test_t6,
}

for name, data in datasets.items():
    data.to_csv(PROCESSED_DIR / f"{name}.csv", index=False)

joblib.dump(result.scaler, PROCESSED_DIR / "scaler_train_fit.joblib")

split_boundaries = {"train_end": SPLIT_TRAIN_END, "val_end": SPLIT_VAL_END}
with open(PROCESSED_DIR / "split_boundaries.json", "w") as f:
    json.dump(split_boundaries, f, indent=2)

print("=== Shapes ===")
for name, data in datasets.items():
    print(f"{name}: {data.shape}")

print("\n=== Columns (train_t1) ===")
print(result.train_t1.columns.tolist())

print("\n=== Appliances present? ===")
for name, data in datasets.items():
    print(f"{name}: {'Appliances' in data.columns}")

print("\n=== Appliances raw vs scaled — spot check ===")
# raw df_features has the untouched original Appliances column
df_features_check = df_raw.sort_values("date").reset_index(drop=True)
raw_lookup = df_features_check.set_index("date")["Appliances"] if "date" in df_features_check else None
sample = result.train_t1.iloc[0]
print(f"train_t1 first row date: {sample['date']}, Appliances value: {sample['Appliances']}")
matching_raw = df_raw.loc[df_raw['date'] == sample['date'], 'Appliances'].values
print(f"raw CSV value at same date: {matching_raw}")
print(f"Match: {sample['Appliances'] == matching_raw[0] if len(matching_raw) else 'N/A'}")

print("\n=== NaN check across all six ===")
for name, data in datasets.items():
    print(f"{name}: {data.isna().sum().sum()} NaNs")

=== Shapes ===
train_t1: (13866, 20)
val_t1: (1728, 20)
test_t1: (3996, 20)
train_t6: (13866, 20)
val_t6: (1728, 20)
test_t6: (3991, 20)

=== Columns (train_t1) ===
['date', 'Appliances_lag_1', 'Appliances_lag_2', 'Appliances_lag_3', 'Appliances_lag_4', 'Appliances_lag_5', 'Appliances_lag_6', 'Appliances_lag_144', 'Appliances_roll6_mean', 'Appliances_roll6_std', 'Appliances_roll18_mean', 'Appliances_roll18_std', 'hour_of_day', 'day_of_week', 'is_weekend', 'minute_of_day_sin', 'minute_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'target_t1']

=== Appliances present? ===
train_t1: False
val_t1: False
test_t1: False
train_t6: False
val_t6: False
test_t6: False

=== Appliances raw vs scaled — spot check ===


KeyError: 'Appliances'

In [12]:
import inspect
from src.pipeline import run_pipeline

print(inspect.getsource(run_pipeline))



def run_pipeline(df_raw: pd.DataFrame) -> PipelineResult:
    """
    Run the complete Phase 2 feature engineering and preprocessing pipeline.

    Feature construction runs on the full continuous time series before
    splitting so lag and rolling features retain temporal context across
    partition boundaries.

    The scaler is fitted only on training rows and then reused unchanged
    for train, validation, and test transformations.

    Final modeling datasets are assembled independently for each horizon
    and partition, and each carries a raw "Appliances" context column
    alongside the engineered FEATURE_COLUMNS — needed by the naive
    persistence baseline (Forecaster.required_columns == ["Appliances"])
    without adding raw current-timestep Appliances to the learned-model
    feature contract itself.
    """

    # 1. Feature engineering on the full continuous series.
    df_features = build_features(
        df_raw,
        target_horizons=TARGET_HORIZONS,
    )

    

In [13]:
import inspect
from src.pipeline import run_pipeline
from src.data.assemble import build_horizon_dataset

print(inspect.signature(build_horizon_dataset))

(df: pandas.DataFrame, feature_columns: list[str], target_column: str, date_column: str = 'date') -> pandas.DataFrame


In [3]:
import pandas as pd
import joblib
import json
from pathlib import Path

from config.paths import RAW_DATA_PATH
from config.features import SPLIT_TRAIN_END, SPLIT_VAL_END
from src.pipeline import run_pipeline
from src.data.assemble import build_horizon_dataset

# sanity check FIRST, before touching real data
import inspect
print("build_horizon_dataset signature:", inspect.signature(build_horizon_dataset))
assert "context_columns" in inspect.signature(build_horizon_dataset).parameters, \
    "STILL STALE — kernel restart did not pick up the edit. Check the file on disk."

df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
result = run_pipeline(df_raw)

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"

datasets = {
    "train_t1": result.train_t1, "val_t1": result.val_t1, "test_t1": result.test_t1,
    "train_t6": result.train_t6, "val_t6": result.val_t6, "test_t6": result.test_t6,
}

for name, data in datasets.items():
    data.to_csv(PROCESSED_DIR / f"{name}.csv", index=False)

joblib.dump(result.scaler, PROCESSED_DIR / "scaler_train_fit.joblib")

split_boundaries = {"train_end": SPLIT_TRAIN_END, "val_end": SPLIT_VAL_END}
with open(PROCESSED_DIR / "split_boundaries.json", "w") as f:
    json.dump(split_boundaries, f, indent=2)

print("\n=== Shapes ===")
for name, data in datasets.items():
    print(f"{name}: {data.shape}")

print("\n=== Columns (train_t1) ===")
print(result.train_t1.columns.tolist())

print("\n=== Appliances present? ===")
for name, data in datasets.items():
    print(f"{name}: {'Appliances' in data.columns}")

print("\n=== Appliances raw match check ===")
sample = result.train_t1.iloc[0]
matching_raw = df_raw.loc[df_raw['date'] == sample['date'], 'Appliances'].values
print(f"train_t1 row0 Appliances: {sample['Appliances']}, raw CSV value: {matching_raw[0]}")
print(f"Match: {sample['Appliances'] == matching_raw[0]}")

print("\n=== NaN check ===")
for name, data in datasets.items():
    print(f"{name}: {data.isna().sum().sum()} NaNs")

build_horizon_dataset signature: (df: pandas.DataFrame, feature_columns: list[str], target_column: str, context_columns: list[str] | None = None, date_column: str = 'date') -> pandas.DataFrame

=== Shapes ===
train_t1: (13866, 21)
val_t1: (1728, 21)
test_t1: (3996, 21)
train_t6: (13866, 21)
val_t6: (1728, 21)
test_t6: (3991, 21)

=== Columns (train_t1) ===
['date', 'Appliances_lag_1', 'Appliances_lag_2', 'Appliances_lag_3', 'Appliances_lag_4', 'Appliances_lag_5', 'Appliances_lag_6', 'Appliances_lag_144', 'Appliances_roll6_mean', 'Appliances_roll6_std', 'Appliances_roll18_mean', 'Appliances_roll18_std', 'hour_of_day', 'day_of_week', 'is_weekend', 'minute_of_day_sin', 'minute_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'Appliances', 'target_t1']

=== Appliances present? ===
train_t1: True
val_t1: True
test_t1: True
train_t6: True
val_t6: True
test_t6: True

=== Appliances raw match check ===
train_t1 row0 Appliances: 60, raw CSV value: 60
Match: True

=== NaN check ===
train_t1: 0

## 10. Carries Into Phase 3
- `FEATURE_COLUMNS` + `target_t1`/`target_t6` are the finalized schema.
- Naive persistence/seasonal baselines must be honestly beaten before Linear Regression; LR + Random Forest before any deep model.
- MLflow logging starts with the first baseline, not the first LSTM.
- **Open item:** scaler is only a local `joblib` file — should become an MLflow artifact tied to its model's run, so model/scaler versions can't drift apart.
- Test suite (25 passing) proves, not just documents, the leakage guarantees: rolling causality (mutate future row → past feature unchanged), scaler isolation (mutate val/test → train scaler stats unchanged), empty-mask guard (raises, not silent).